## Before you start

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [ ]:
!nvidia-smi

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

## Install YOLOv8

YOLOv8 can be installed in two ways - from the source and via pip. This is because it is the first iteration of YOLO to have an official package.

In [ ]:
# Pip install method (recommended)

!pip install ultralytics==8.2.103 -q

from IPython import display
display.clear_output()

import ultralytics
ultralytics.checks()

In [ ]:
from ultralytics import YOLO

from IPython.display import display, Image

🟢 Tip: The examples below work even if you use our non-custom model. However, you won't be able to deploy it to Roboflow. To do that, create a custom dataset as described below or fork (copy) one into your workspace from Universe.

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="6CNEVnSz8AE7UYkyS9gU")
project = rf.workspace("shantanu-maity").project("potholes-detection-qwkkc")
version = project.version(10)
dataset = version.download("yolov8")


## Custom Training

In [ ]:
%cd {HOME}

!yolo task=detect mode=train model=yolov8s.pt data=/content/Potholes-Detection-10/data.yaml epochs=10 imgsz=640 plots=True

In [ ]:
%cd {HOME}
Image(filename=f'{HOME}/runs/detect/train/confusion_matrix.png', width=600)

In [ ]:
%cd {HOME}
Image(filename=f'{HOME}/runs/detect/train/results.png', width=600)

In [ ]:
%cd {HOME}
Image(filename=f'{HOME}/runs/detect/train/val_batch0_pred.jpg', width=600)

## Validate Custom Model

In [ ]:
%cd {HOME}

!yolo task=detect mode=val model={HOME}/runs/detect/train/weights/best.pt data={dataset.location}/data.yaml

## Inference with Custom Model

In [ ]:
%cd {HOME}
!yolo task=detect mode=predict model={HOME}/runs/detect/train/weights/best.pt conf=0.25 source={dataset.location}/test/images save=True

**NOTE:** Let's take a look at few results.

In [ ]:
import glob
from IPython.display import Image, display

# Define the base path where the folders are located
base_path = '/content/runs/detect/'

# List all directories that start with 'predict' in the base path
subfolders = [os.path.join(base_path, d) for d in os.listdir(base_path)
              if os.path.isdir(os.path.join(base_path, d)) and d.startswith('predict')]

# Find the latest folder by modification time
latest_folder = max(subfolders, key=os.path.getmtime)

image_paths = glob.glob(f'{latest_folder}/*.jpg')[:3]

# Display each image
for image_path in image_paths:
    display(Image(filename=image_path, width=600))
    print("\n")

In [ ]:
# Load the trained model
model = YOLO("/content/runs/detect/train/weights/best.pt")

# Evaluate on a test set
results = model.val(data="/content/Potholes-Detection-10/data.yaml")

# Inspect the structure of results
print("Results object:", results)
print("Attributes and methods of results:", dir(results))

# Check if results is a dictionary-like object
if isinstance(results, dict):
    print("Keys in results:", results.keys())
else:
    # If it's not a dictionary, try checking for other common attribute names
    print("Contents of 'results':", results.__dict__ if hasattr(results, '__dict__') else "No __dict__ attribute")

# Once you understand the structure, try accessing metrics again based on the actual structure


In [ ]:
!yolo task=detect mode=predict model= "/content/runs/detect/train/weights/best.pt" conf=0.25 source=test.jpeg save=True

In [ ]:
Image("/content/runs/detect/predict2/test.jpeg", width=600)

In [ ]:
!yolo task=detect mode=predict model= "/content/runs/detect/train/weights/best.pt" conf=0.25 source="p.mp4" save=True

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import os

# Input video path
save_path = '/content/runs/detect/predict3/p.avi'

# Compressed video path
compressed_path = "/content/result_compressed.mp4"

os.system(f"ffmpeg -i {save_path} -vcodec libx264 {compressed_path}")

# Show video
mp4 = open(compressed_path,'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=400 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

In [ ]:
from ultralytics import YOLO

# Load your trained model
model = YOLO("/content/runs/detect/train/weights/best.pt")  # Replace with your model's path

# Evaluate the model on the test set
results = model.val(data="/content/Potholes-Detection-10/data.yaml")  # Replace with your dataset's path

# Display results
metrics = results.results_dict  # Access metrics in dictionary format
print("Evaluation Metrics:")
print(f"Precision: {metrics['metrics/precision(B)']:.4f}")
print(f"Recall: {metrics['metrics/recall(B)']:.4f}")
print(f"mAP@0.5: {metrics['metrics/mAP50(B)']:.4f}")
print(f"mAP@0.5:0.95: {metrics['metrics/mAP50-95(B)']:.4f}")